# Local analysis

Tier 1 ACL evaluation: per-condition summary, relative WER gain vs `arm_a`, and figures.
Reads `eval_results/tier1_acl_results.csv` and writes figures to `figures/tier1/`.
Run from the repo root.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

FIGURES = Path("figures/tier1")
FIGURES.mkdir(parents=True, exist_ok=True)
RESULT = Path("eval_results/tier1_acl_results.csv").resolve()

if not RESULT.exists():
    print(f"Tier 1 results not found: {RESULT}")
else:
    t1 = pd.read_csv(RESULT)
    print("TIER 1 rows:", len(t1), "\n")

    summary = (
        t1.groupby("condition", dropna=False)[["wer", "cer", "rtf", "seconds", "signal_snr_improvement_db", "term_presence_recall"]]
           .agg(["mean", "min", "max", "median"])
           .round(5)
    )
    print("OVERALL per condition:")
    print(summary)

    print("\nRELATIVE ASR GAIN vs arm_a, grouped by (noise_type, snr_db):")
    for key, sub in t1.groupby(["noise_type", "snr_db"]):
        base = sub.loc[sub.condition.eq("arm_a"), "wer"].mean() if (sub.condition == "arm_a").any() else float("nan")
        if pd.isna(base) or base <= 0:
            continue
        deltas = (sub.groupby("condition")["wer"].mean() - base) / base * 100.0
        deltas["__arm_a_baseline__"] = 0.0
        print(f"\n-- {key} -- arm_a mean WER = {base:.4f}")
        print(deltas.sort_values().round(2).to_string())

    # Figure 1: mean WER per condition.
    means = t1.groupby("condition", sort=True)["wer"].mean()
    errs = t1.groupby("condition", sort=True)["wer"].std(ddof=1).fillna(0.0)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(means.index, means.values, yerr=errs.values, capsize=4, color="#4c78a8")
    ax.set_title("Tier 1  mean WER by DSP condition")
    ax.set_xlabel("DSP condition")
    ax.set_ylabel("WER (lower is better)")
    ax.set_xticks(range(len(means.index.tolist())))
    ax.set_xticklabels(means.index.tolist(), rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier1_wer_by_condition.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 2: relative WER gain vs arm_a under seeded DEMAND noise.
    baselines = t1.loc[t1.condition == "arm_a", ["noise_type", "snr_db", "talk_id", "wer"]]
    baselines = baselines.rename(columns={"wer": "arm_a_wer"})
    delta_rows = t1.merge(baselines, on=["noise_type", "snr_db", "talk_id"], how="inner")
    delta_rows["delta_vs_arm_a"] = (delta_rows["wer"] - delta_rows["arm_a_wer"]) / delta_rows["arm_a_wer"].clip(lower=1e-9) * 100.0
    delta_mean = delta_rows.groupby("condition")["delta_vs_arm_a"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(delta_mean.index, delta_mean.values, color="#d6604d")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Tier 1  mean WER change vs arm_a")
    ax.set_xlabel("DSP condition")
    ax.set_ylabel("Relative WER change (%)")
    ax.set_xticks(range(len(delta_mean.index.tolist())))
    ax.set_xticklabels(delta_mean.index.tolist(), rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier1_wer_gain_vs_arm_a.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 3: SNR improvement vs WER change.
    noisy = delta_rows.loc[delta_rows.noise_type != "none", :]
    if "signal_snr_improvement_db" in noisy.columns and noisy["signal_snr_improvement_db"].notna().any():
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.scatter(noisy["signal_snr_improvement_db"], noisy["delta_vs_arm_a"], alpha=0.45)
        ax.set_title("Tier 1  signal SNR improvement vs WER change")
        ax.set_xlabel("Signal SNR improvement (dB)")
        ax.set_ylabel("WER change vs arm_a (%)")
        fig.tight_layout()
        fig.savefig(FIGURES / "tier1_snr_improvement_vs_wer.png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        print("Saved Tier 1 figures to", FIGURES)
    else:
        print("No signal SNR improvement values found; skipped Tier 1 SNR figure")

    # Figure 4: RTF vs WER tradeoff.
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(delta_rows.groupby("condition")[["wer", "rtf"]].mean()["rtf"].values,
               delta_rows.groupby("condition")[["wer", "rtf"]].mean()["wer"].values)
    for cond, point in delta_rows.groupby("condition")[["wer", "rtf"]].mean().iterrows():
        ax.annotate(cond, (point["rtf"], point["wer"]))
    ax.set_title("Tier 1  RTF vs WER tradeoff")
    ax.set_xlabel("RTF (higher = more compute)")
    ax.set_ylabel("WER")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier1_rtf_vs_wer.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved Tier 1 figures to", FIGURES)